In [1]:
# Step 0: Import Libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: Set display options
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

#pandas - reading csv files, cleaning / transforming data (data manipulation and analysis)
#numercial operations and handling arrays
#scipy - advanced statistics (z-score cal, hyptohesis testing, outlier detection, probability)
#matplotlib - data visualization - line plots, bar charts, scatter plots..)
#seaborn - top of matplotlib , statistical visualization - boxplots, histograms, heatmaps...

In [2]:
# Step 1: Load Data
df = pd.read_csv('D:\\Phani\\etechguild\\syllabus\\demo\\lessons\\python\\EDA\\Titanic-Dataset.csv')  # Replace with your file path
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.00,1,0,A/5 21171,7.25,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.00,1,0,PC 17599,71.28,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.00,0,0,STON/O2. 3101282,7.92,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.00,1,0,113803,53.10,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.00,0,0,373450,8.05,NaN,S


In [3]:
#  Step 2: Identify Missing Values
def summarize_missing(df):
    missing_summary = df.isnull().sum()
    missing_percent = (missing_summary / len(df)) * 100
    return pd.DataFrame({'Missing Count': missing_summary, 'Missing %': missing_percent}).sort_values(by='Missing %', ascending=False)

summarize_missing(df)


,Missing Count,Missing %
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22
PassengerId,0,0.00
Survived,0,0.00
Pclass,0,0.00
Name,0,0.00
Sex,0,0.00
SibSp,0,0.00
Parch,0,0.00


In [4]:
# Step 3: Convert disguised missing values to NaN
def convert_disguised_missing(df, missing_indicators=["NA", "XX", "999", ""]):
    return df.replace(missing_indicators, np.nan)

df = convert_disguised_missing(df)


In [5]:
#  Step 4: Fill missing values safely
def fill_missing(df, strategy='median', columns=[]):
    for col in columns:
        if strategy == 'median':
            df[col] = df[col].fillna(df[col].median())
        elif strategy == 'mean':
            df[col] = df[col].fillna(df[col].mean())
        elif strategy == 'mode':
            df[col] = df[col].fillna(df[col].mode()[0])
    return df

df = fill_missing(df, strategy='median', columns=['Age'])  # Example

#mode - best for categorical / discrete - non-numeric
#mean - numeric 
#median - ex: age column / robust to outliers 


In [6]:
# Step 5: Drop rows/columns with excessive missingness
def drop_excess_missing(df, row_thresh=0.5, col_thresh=0.5):
    df = df.dropna(thresh=int((1 - row_thresh) * df.shape[1]))
    df = df.loc[:, df.isnull().mean() < col_thresh]
    return df

df = drop_excess_missing(df)

#row_thresh=0.5 , drop rows with more than 50% missing values
#df.shape[1] is the number of columns
#df.isnull().mean() gives the percentage of missing values per column
    # <col_thresh - keeps only columns where missing values is less than col thresh of 50%

In [7]:
# Step 6: Standardise Units and Precision
def convert_units(df, col, factor, new_col=None, round_to=2):
    new_col = new_col or col
    df[new_col] = (df[col] * factor).round(round_to)
    return df

# Example: Convert lbs to kg
# df = convert_units(df, col='Weight_lbs', factor=0.453592, new_col='Weight_kg')

# col : the column with values in the original unit 
#factor : the conversion multiplier 
#new_col - optional - name of the new column to store converted values. If not provided, it overwrites the original 

In [8]:
#Step 7: Remove Outliers
def remove_outliers(df, col, method='zscore', threshold=3):
    if method == 'zscore':
        return df[np.abs(stats.zscore(df[col].dropna())) < threshold]
    elif method == 'iqr':
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        return df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]
    return df

# df = remove_outliers(df, col='Fare', method='iqr')

#iqr - removes the values that are outside of the range

In [9]:
# Step 8: Standardise Text
def clean_text_column(df, col):
    df[col] = df[col].astype(str).str.strip().str.title()
    return df

df = clean_text_column(df, 'Name')  # Example


In [10]:
# Step 9: Standardise Date Formats
def standardize_dates(df, col, format='%d/%m/%y'):
    df[col] = pd.to_datetime(df[col], format=format, errors='coerce')
    return df

# df = standardize_dates(df, 'DateOfBirth')

#errors='coerce' - if a value doesn't match the format, it's replaced with NaT (not a time)

In [11]:
#  Step 10: Final Check
print("Final Missing Summary:")
print(summarize_missing(df))

print("\nData Types:")
print(df.dtypes)

df.head()


Final Missing Summary:
             Missing Count  Missing %
Embarked                 2       0.22
PassengerId              0       0.00
Survived                 0       0.00
Pclass                   0       0.00
Name                     0       0.00
Sex                      0       0.00
Age                      0       0.00
SibSp                    0       0.00
Parch                    0       0.00
Ticket                   0       0.00
Fare                     0       0.00

Data Types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Embarked        object
dtype: object


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.00,1,0,A/5 21171,7.25,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.00,1,0,PC 17599,71.28,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.00,0,0,STON/O2. 3101282,7.92,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.00,1,0,113803,53.10,S
4,5,0,3,"Allen, Mr. William Henry",male,35.00,0,0,373450,8.05,S
